# Task 5 — Embeddings & ChromaDB Knowledge Base (token-aware)

Builds the searchable knowledge base from the token-aware chunks (Task 4 branch pipeline) and verifies it with sample semantic-search queries. Implementation lives in `src/embedding.py` and `src/knowledge_base.py` (tests in `tests/`).

Key design points:
- **Embedding model = `BAAI/bge-small-en-v1.5`** (384-dim, 512-token input limit) — the same model whose tokenizer sizes the chunks, so **every chunk fits the model's input window** and nothing is silently truncated at embedding time.
- **One ChromaDB collection per chunking config** (`papers_<config_id>`) — different chunk-size/overlap experiments can coexist and be compared by retrieval quality without mixing.
- **Citations built in** — every result carries paper id, title, and page range, ready for the memo-drafting step and the November page-level retrieval evaluation.
- **Swappable input** — currently chunks the interim-cleaned curated dataset; switches to the team's canonical cleaned JSON once one is agreed.

In [1]:
import sys
sys.path.insert(0, "..")

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import pandas as pd

from src.data_io import load_pages
from src.cleaning import clean_pages
from src.chunking import ChunkingConfig, build_chunks
from src.knowledge_base import build_knowledge_base, search

config = ChunkingConfig()  # 512 tokens, 64 overlap, page scope
chunks = build_chunks(clean_pages(load_pages()), config)
print(f"{len(chunks)} chunks ready to embed")

553 chunks ready to embed


## Build the knowledge base

Embeds all chunks and upserts them into a persistent ChromaDB at `../chroma/` (gitignored — rebuilt locally in ~a minute).

In [2]:
collection = build_knowledge_base(chunks, path="../chroma")
print("collection:", collection.name)
print("vectors stored:", collection.count())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

collection: papers_dcd1d776
vectors stored: 553


## Sample semantic-search queries

One targeted question per paper, checking the top retrieved chunk comes from the expected paper — a sanity check of the whole pipeline (cleaning → chunking → embedding → retrieval), not the official evaluation (that runs in November against the provided ground-truth Q&A pairs).

In [3]:
queries = [
    ("How does LoRA reduce the number of trainable parameters during fine-tuning?", "2106.09685"),
    ("How does FlashAttention reduce reads and writes between GPU HBM and SRAM?", "2205.14135"),
    ("Where in the context window do language models struggle to use relevant information?", "2307.03172"),
    ("How is the dense passage retriever trained using in-batch negatives?", "2004.04906"),
    ("How does Self-RAG use reflection tokens to decide when to retrieve?", "2310.11511"),
    ("How does retrieval-augmented generation combine a retriever with a seq2seq generator?", "2005.11401"),
]

rows = []
for query, expected in queries:
    top = search(collection, query, k=3)
    rows.append({
        "query": query[:58] + ("…" if len(query) > 58 else ""),
        "expected": expected,
        "top paper": top[0]["paper_id"],
        "hit": top[0]["paper_id"] == expected,
        "pages": f"{top[0]['page_start']}-{top[0]['page_end']}",
        "similarity": top[0]["similarity"],
    })
df = pd.DataFrame(rows)
print(f"top-1 correct paper: {df['hit'].sum()}/{len(df)}")
df

top-1 correct paper: 5/6


,query,expected,top paper,hit,pages,similarity
0,How does LoRA reduce the number of trainable p...,2106.09685,2106.09685,True,8-8,0.7968
1,How does FlashAttention reduce reads and write...,2205.14135,2205.14135,True,2-2,0.8505
2,Where in the context window do language models...,2307.03172,2307.03172,True,2-2,0.8401
3,How is the dense passage retriever trained usi...,2004.04906,2004.04906,True,6-6,0.8673
4,How does Self-RAG use reflection tokens to dec...,2310.11511,2312.10997,False,12-12,0.8088
5,How does retrieval-augmented generation combin...,2005.11401,2005.11401,True,2-2,0.8405


## What a retrieved result looks like

Each result already carries what a cited memo needs: the passage, the paper, and the page range.

In [4]:
example = search(collection, "How does LoRA reduce the number of trainable parameters during fine-tuning?", k=1)[0]
print(f"{example['paper_title']}")
print(f"pages {example['page_start']}-{example['page_end']} | similarity {example['similarity']} | {example['chunk_id']}")
print()
print(example["text"][:900])

LoRA: Low-Rank Adaptation of Large Language Models
pages 8-8 | similarity 0.7968 | 2106.09685_dcd1d776_c0021

opposed to providing one for every entry. See Section D.4 for details on the hyperparameters used.
As shown in Table 4, LoRA matches or exceeds the fine-tuning baseline on all three datasets. Note
that not all methods benefit monotonically from having more trainable parameters, as shown in Fig-ure 2. We observe a significant performance drop when we use more than 256 special tokens for
prefix-embedding tuning or more than 32 special tokens for prefix-layer tuning. This corroborates
similar observations in Li & Liang (2021). While a thorough investigation into this phenomenon
is out-of-scope for this work, we suspect that having more special tokens causes the input distribution to shift further away from the pre-training data distribution. Separately, we investigate the
performance of different adaptation approaches in the low-data regime in Section F.3.
6 7 8 9 10 11
log10 # Tr

## Measured retrieval quality, and reranking (stretch goal #2 groundwork)

`src/retrieval_eval.py` computes deterministic hit rates (no LLM-judge, no API calls) — the same function the November evaluation will use with the official ground-truth Q&A pairs. `search(..., rerank=True)` re-scores candidates with a local FlashRank cross-encoder before returning the top k.

In [5]:
from src.retrieval_eval import evaluate_retrieval

reports = [evaluate_retrieval(collection, queries, ks=(1, 3), rerank=flag) for flag in (False, True)]
pd.DataFrame([{"rerank": r["rerank"], "hit@1": r["hit@1"], "hit@3": r["hit@3"],
               "misses@1": [m["expected"] for m in r["misses_at_1"]]} for r in reports])

,rerank,hit@1,hit@3,misses@1
0,False,0.833,1.0,[2310.11511]
1,True,0.833,1.0,[2310.11511]


## Takeaways

- End-to-end pipeline works: cleaning → token-aware chunking → embedding → cited retrieval, all from importable `src/` modules (the October RAG pipeline can `from src.knowledge_base import search`).
- Every stored vector represents its chunk's **full text** — chunks are sized with the embedding model's own tokenizer, so nothing exceeds the model's 512-token window.
- Measured on the sample queries: **hit@1 = 5/6, hit@3 = 6/6** — the correct paper is always in the top 3, which is what the memo generator will consume. The one rank-1 miss retrieves the RAG survey's genuine coverage of Self-RAG.
- Local cross-encoder reranking (FlashRank) is wired in behind `search(..., rerank=True)`; on 6 queries it neither helps nor hurts — its real value gets judged on the ground-truth Q&A set in November, using the same `evaluate_retrieval` on per-config collections.
- The collection stores which embedding model built it, and `search` refuses a mismatched model — mixed embedding spaces fail loudly instead of returning garbage.
- Input is swappable: when the team settles on a canonical cleaned JSON, only the loading step changes (`src/data_io.py` already has the adapter).